# 05_manuscript_repository_and_submission_packaging.ipynb

Packages Notebook 04 outputs for repository upload and manuscript-support submission files.

Required input: `outputs 4.zip` from Notebook 04.


In [1]:
%pip install -U pandas numpy tabulate openpyxl


Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os, json, zipfile, shutil
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd

BASE_DIR = Path("sodium_cathode_submission_package")
INPUT_DIR = BASE_DIR / "input"
OUTPUT_DIR = BASE_DIR / "outputs"
REPOSITORY_DIR = OUTPUT_DIR / "repository_package"
MANUSCRIPT_DIR = OUTPUT_DIR / "manuscript_package"
SUBMISSION_DIR = OUTPUT_DIR / "submission_files"
AUDIT_DIR = OUTPUT_DIR / "audit"
for d in [INPUT_DIR, OUTPUT_DIR, REPOSITORY_DIR, MANUSCRIPT_DIR, SUBMISSION_DIR, AUDIT_DIR]:
    d.mkdir(parents=True, exist_ok=True)
print("Base folder:", BASE_DIR.resolve())


Base folder: C:\Users\IICT-1\next sustainability\Final for github\sodium_cathode_submission_package


In [3]:
def find_first_existing(candidates):
    for p in candidates:
        p = Path(p)
        if p.exists():
            return p
    raise FileNotFoundError("Could not find any of:\n" + "\n".join(map(str, candidates)))

def find_inside(root, filename):
    matches = list(Path(root).rglob(filename))
    if not matches:
        raise FileNotFoundError(f"Missing {filename} inside {root}")
    return matches[0]

OUTPUTS4_ZIP = find_first_existing([
    "outputs 4.zip", "outputs_4.zip", "outputs4.zip",
    "/mnt/data/outputs 4.zip", "/mnt/data/outputs_4.zip", "/mnt/data/outputs4.zip"
])
EXTRACT_DIR = INPUT_DIR / "notebook04_outputs_extracted"
if EXTRACT_DIR.exists():
    shutil.rmtree(EXTRACT_DIR)
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(OUTPUTS4_ZIP) as z:
    z.extractall(EXTRACT_DIR)
final_audits = list(EXTRACT_DIR.rglob("notebook04_final_output_audit.csv"))
if not final_audits:
    raise FileNotFoundError("Could not find Notebook 04 final audit in outputs 4.zip")
SOURCE_OUTPUTS_DIR = final_audits[0].parent.parent
print("Loaded:", OUTPUTS4_ZIP)
print("Using Notebook 04 outputs:", SOURCE_OUTPUTS_DIR)


Loaded: outputs 4.zip
Using Notebook 04 outputs: sodium_cathode_submission_package\input\notebook04_outputs_extracted\outputs


In [4]:
PATHS = {
    "input_audit": find_inside(SOURCE_OUTPUTS_DIR, "notebook04_input_audit.csv"),
    "final_audit": find_inside(SOURCE_OUTPUTS_DIR, "notebook04_final_output_audit.csv"),
    "table1": find_inside(SOURCE_OUTPUTS_DIR, "table_1_dataset_funnel.csv"),
    "table2": find_inside(SOURCE_OUTPUTS_DIR, "table_2_ml_grouped_cv_summary_manuscript.csv"),
    "table3": find_inside(SOURCE_OUTPUTS_DIR, "table_3_family_level_summary_manuscript.csv"),
    "table4": find_inside(SOURCE_OUTPUTS_DIR, "table_4_top20_candidate_shortlist_manuscript.csv"),
    "feature_importance": find_inside(SOURCE_OUTPUTS_DIR, "table_feature_importance_top20_by_target.csv"),
    "candidate_map": find_inside(SOURCE_OUTPUTS_DIR, "figure_data_candidate_voltage_capacity_energy_map.csv"),
    "framing": find_inside(SOURCE_OUTPUTS_DIR, "manuscript_framing_methods_results_limitations.md"),
}

df_input_audit = pd.read_csv(PATHS["input_audit"])
df_final_audit = pd.read_csv(PATHS["final_audit"])
df_funnel = pd.read_csv(PATHS["table1"])
df_ml = pd.read_csv(PATHS["table2"])
df_family = pd.read_csv(PATHS["table3"])
df_top20 = pd.read_csv(PATHS["table4"])
df_feature_importance = pd.read_csv(PATHS["feature_importance"])
df_candidate_map = pd.read_csv(PATHS["candidate_map"])
framing_text = Path(PATHS["framing"]).read_text(encoding="utf-8").replace("\\n\\n", "\n\n")

if "exists" in df_final_audit.columns:
    missing = df_final_audit[df_final_audit["exists"].astype(str).str.lower() != "true"]
    assert len(missing) == 0, "Some Notebook 04 outputs are missing"

audit_dict = dict(zip(df_input_audit["check"], df_input_audit["value"]))
assert int(audit_dict.get("master_rows", -1)) == 416
assert int(audit_dict.get("unique_mp_index", -1)) == 416
assert int(audit_dict.get("duplicate_mp_index", -1)) == 0
assert int(audit_dict.get("candidate_all_rows", -1)) == 112
assert int(audit_dict.get("candidate_unique_rows", -1)) == 95
assert int(audit_dict.get("family_summary_total_records", -1)) == 416
print("Notebook 04 outputs validated successfully.")
display(df_input_audit)


Notebook 04 outputs validated successfully.


,check,value
0,master_rows,416
1,unique_mp_index,416
2,duplicate_mp_index,0
3,candidate_all_rows,112
4,candidate_unique_rows,95
5,family_summary_total_records,416
6,ml_summary_rows,12
7,protocol_A_feature_count,61
8,protocol_B_feature_count,68
9,protocol_C_feature_count,73


In [5]:
# EDIT THESE PLACEHOLDERS BEFORE FINAL REPOSITORY/SUBMISSION UPLOAD
PROJECT_TITLE = "Criticality-aware materials-informatics screening of sodium-ion cathode materials using computed electrode descriptors and text-mined literature evidence"
SHORT_TITLE = "Sustainable sodium-ion cathode screening"
TARGET_JOURNAL = "Next Sustainability"
ARTICLE_TYPE = "Research Article"
AUTHORS = ["Author One", "Author Two"]
AFFILIATIONS = ["Department / Institute, University, City, Country"]
CORRESPONDING_AUTHOR = "Author One"
CORRESPONDING_EMAIL = "your.email@example.com"
KEYWORDS = ["Sodium-ion batteries", "Cathode materials", "Materials informatics", "Sustainable materials", "Machine learning", "Materials Project", "Text-mined literature evidence"]
REPOSITORY_URL = "TO_BE_ADDED_AFTER_GITHUB_OR_ZENODO_UPLOAD"
DATA_REPOSITORY_DOI = "TO_BE_ADDED_AFTER_ZENODO_OR_FIGSHARE_UPLOAD"
TODAY = datetime.now().strftime("%Y-%m-%d")

def get_audit(name, default=np.nan):
    return audit_dict.get(name, default)

n_master = int(get_audit("master_rows"))
n_candidate_all = int(get_audit("candidate_all_rows"))
n_candidate_unique = int(get_audit("candidate_unique_rows"))
n_protocol_a = int(get_audit("protocol_A_feature_count"))
n_protocol_b = int(get_audit("protocol_B_feature_count"))
n_protocol_c = int(get_audit("protocol_C_feature_count"))

def funnel_count(term):
    row = df_funnel[df_funnel["stage"].astype(str).str.contains(term, case=False, na=False)]
    return int(row.iloc[0]["count"]) if len(row) else np.nan

n_basic = funnel_count("Basic electrochemical")
n_hard = funnel_count("Hard-exclusion-free")
n_cons = funnel_count("Conservative earth-abundant")
n_cons_cde = funnel_count("Conservative candidates with exact CDE")
print(PROJECT_TITLE)
print("Key counts:", n_master, n_basic, n_hard, n_cons, n_cons_cde, n_candidate_unique)


Criticality-aware materials-informatics screening of sodium-ion cathode materials using computed electrode descriptors and text-mined literature evidence
Key counts: 416 164 112 56 18 95


In [6]:
# Create repository core files
REPOSITORY_DIR.mkdir(parents=True, exist_ok=True)
readme = f"""# {PROJECT_TITLE}

## Overview
This repository contains a reproducible materials-informatics workflow for sustainability-aware screening of sodium-ion battery cathode materials.

The workflow combines Materials Project sodium-ion electrode records, conservative sustainability filtering, ChemDataExtractor-based text-mined evidence matching, grouped-CV machine learning, and manuscript-ready candidate shortlists.

## Dataset summary
| Item | Count |
|---|---:|
| Materials Project sodium-ion electrode records | {n_master} |
| Basic electrochemical screening pass | {n_basic} |
| Hard-exclusion-free candidates | {n_hard} |
| Conservative earth-abundant candidates | {n_cons} |
| Conservative candidates with exact CDE evidence | {n_cons_cde} |
| All candidate shortlist records | {n_candidate_all} |
| Unique manuscript shortlist entries | {n_candidate_unique} |

## Descriptor protocols
| Protocol | Description | Feature count |
|---|---|---:|
| Protocol A | Composition-only descriptors | {n_protocol_a} |
| Protocol B | Composition + lattice/structure descriptors | {n_protocol_b} |
| Protocol C-strict | Composition + lattice/structure + post-DFT electrode descriptors | {n_protocol_c} |

Protocol C-strict is a post-computation decision-support model, not a pre-DFT discovery model.

## Run order
1. 01_mp_sodium_cathode_data_extraction.ipynb
2. 02_cde_literature_evidence_matching.ipynb
3. 03_mp_sodium_cathode_ml_protocols_FINAL_FIXED.ipynb
4. 04_figures_tables_and_manuscript_framing.ipynb
5. 05_manuscript_repository_and_submission_packaging.ipynb

## Claim control
CDE evidence is text-mined literature evidence only. The shortlist is a decision-support ranking, not experimental validation.

## Repository URL
{REPOSITORY_URL}

## Data DOI
{DATA_REPOSITORY_DOI}
"""
(REPOSITORY_DIR / "README.md").write_text(readme, encoding="utf-8")
(REPOSITORY_DIR / "requirements.txt").write_text("pandas\nnumpy\nscikit-learn\nscipy\nplotly\ntabulate\nopenpyxl\npymatgen\nmp-api\ntqdm\njoblib\n", encoding="utf-8")
(REPOSITORY_DIR / "environment.yml").write_text("""name: sodium-cathode-screening
channels:
  - conda-forge
  - defaults
dependencies:
  - python>=3.10
  - pandas
  - numpy
  - scikit-learn
  - scipy
  - plotly
  - tabulate
  - openpyxl
  - tqdm
  - joblib
  - pip
  - pip:
      - pymatgen
      - mp-api
""", encoding="utf-8")
(REPOSITORY_DIR / "RUN_ORDER.md").write_text("""# Notebook run order

1. `01_mp_sodium_cathode_data_extraction.ipynb`
2. `02_cde_literature_evidence_matching.ipynb`
3. `03_mp_sodium_cathode_ml_protocols_FINAL_FIXED.ipynb`
4. `04_figures_tables_and_manuscript_framing.ipynb`
5. `05_manuscript_repository_and_submission_packaging.ipynb`

Do not upload API keys. Use only fixed outputs for final manuscript/repository files.
""", encoding="utf-8")
raw_dir = REPOSITORY_DIR / "data" / "raw"
raw_dir.mkdir(parents=True, exist_ok=True)
(raw_dir / "README_raw_data.md").write_text("""# Raw data note

Raw Materials Project data should be regenerated using the Materials Project API. Do not commit API keys or credential files.

The ChemDataExtractor battery database should be obtained from its original public source.
""", encoding="utf-8")
print("Repository core files created.")


Repository core files created.


In [7]:
# Create manuscript package files
MANUSCRIPT_DIR.mkdir(parents=True, exist_ok=True)
abstract = f"""# Draft abstract

Sodium-ion batteries are promising for sustainable grid-scale and low-cost energy storage because they can reduce dependence on lithium and other critical elements. Here, we present a reproducible materials-informatics workflow for criticality-aware screening of sodium-ion cathode materials using Materials Project sodium-ion electrode records and text-mined battery literature evidence. Starting from {n_master} computed sodium-ion electrode records, {n_basic} passed basic electrochemical screening, {n_hard} were hard-exclusion-free, and {n_cons} satisfied a conservative earth-abundance definition. Exact formula-level ChemDataExtractor evidence was found for {n_cons_cde} conservative candidates, and the deduplicated manuscript shortlist contained {n_candidate_unique} unique entries. Grouped cross-validation compared composition-only, composition-plus-structure, and post-DFT descriptor protocols. The C-strict protocol gave useful predictive performance for voltage, capacity, and gravimetric energy, while stability prediction was weaker and is interpreted cautiously. This study provides a transparent decision-support pipeline rather than experimental validation.
"""
outline = f"""# Manuscript outline for {TARGET_JOURNAL}

## Title
{PROJECT_TITLE}

## Sections
1. Introduction
2. Data sources: Materials Project and ChemDataExtractor
3. Candidate screening strategy
4. Machine-learning protocols and grouped validation
5. Results and discussion
6. Limitations and claim control
7. Conclusion

## Key numbers
- MP records: {n_master}
- Basic screening pass: {n_basic}
- Hard-exclusion-free candidates: {n_hard}
- Conservative candidates: {n_cons}
- Conservative candidates with exact CDE evidence: {n_cons_cde}
- Unique manuscript shortlist: {n_candidate_unique}
"""
fig_caps = f"""# Figure captions

## Figure 1
Reproducible workflow for sustainability-aware sodium-ion cathode screening.

## Figure 2
Candidate screening funnel from {n_master} MP sodium-ion electrode records to {n_candidate_unique} unique manuscript shortlist entries.

## Figure 3
Family-level screening and evidence summary.

## Figure 4
Grouped cross-validation performance across descriptor protocols.

## Figure 5
Candidate voltage-capacity-energy landscape.

## Figure 6
Feature-importance summary for C-strict models.
"""
table_caps = """# Table captions

## Table 1
Dataset and candidate filtering funnel.

## Table 2
Grouped cross-validation model performance.

## Table 3
Family-level summary of screening and evidence.

## Table 4
Top 20 candidate shortlist.
"""
claim = """# Claim-control notes

Safe claims:
- decision-support workflow
- computed sodium-ion electrode records
- text-mined literature evidence
- candidate prioritization
- screening-based shortlist
- post-DFT decision-support model

Avoid:
- experimentally validated candidates
- new cathode discovered
- CDE confirms performance
- ML discovers materials
- guaranteed sustainable
- ready for commercialization
"""
title_page = f"""# Title page draft

## Title
{PROJECT_TITLE}

## Authors
{chr(10).join(['- ' + a for a in AUTHORS])}

## Affiliations
{chr(10).join(['- ' + a for a in AFFILIATIONS])}

## Corresponding author
{CORRESPONDING_AUTHOR}, {CORRESPONDING_EMAIL}

## Keywords
{', '.join(KEYWORDS)}
"""
for name, text in [("draft_abstract.md", abstract), ("manuscript_outline.md", outline), ("figure_captions.md", fig_caps), ("table_captions.md", table_caps), ("claim_control_notes.md", claim), ("title_page_draft.md", title_page)]:
    (MANUSCRIPT_DIR / name).write_text(text, encoding="utf-8")
print("Manuscript package files created.")


Manuscript package files created.


In [8]:
# Create submission declarations
SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)
files = {
    "data_availability_statement.md": f"# Data availability statement\n\nThe processed datasets, candidate screening tables, figure-data files, and analysis outputs generated in this study will be made available at: {DATA_REPOSITORY_DOI}\n\nMaterials Project source data can be accessed through the Materials Project API subject to Materials Project terms. ChemDataExtractor records should be obtained from their original public source.\n",
    "code_availability_statement.md": f"# Code availability statement\n\nThe analysis code and reproducible notebooks used in this study will be made available at: {REPOSITORY_URL}\n",
    "competing_interests_statement.md": "# Declaration of competing interests\n\nThe authors declare no known competing interests.\n",
    "funding_statement.md": "# Funding statement\n\nThis research did not receive any specific grant. Edit if funding exists.\n",
    "ethics_statement.md": "# Ethics statement\n\nThis study uses public computational and text-mined data and does not require ethics approval.\n",
    "generative_ai_declaration.md": "# Generative AI declaration\n\nThe authors used ChatGPT to assist with code structuring, manuscript organization, language refinement, and reproducibility checklist preparation. The authors reviewed and edited the output and take full responsibility for the manuscript. Edit according to actual use.\n",
    "credit_author_contributions.md": "# CRediT author contribution statement\n\nEdit before submission.\n",
}
for name, text in files.items():
    (SUBMISSION_DIR / name).write_text(text, encoding="utf-8")
print("Submission statements created.")


Submission statements created.


In [9]:
# Copy tables, figure data, docs, and notebooks into repository package
repo_data = REPOSITORY_DIR / "data" / "processed"
repo_fig = REPOSITORY_DIR / "data" / "figure_data"
repo_docs = REPOSITORY_DIR / "docs"
repo_nbs = REPOSITORY_DIR / "notebooks"
for d in [repo_data, repo_fig, repo_docs, repo_nbs]:
    d.mkdir(parents=True, exist_ok=True)
for src in [PATHS["table1"], PATHS["table2"], PATHS["table3"], PATHS["table4"], PATHS["feature_importance"]]:
    shutil.copy2(src, repo_data / Path(src).name)
for src in Path(SOURCE_OUTPUTS_DIR).rglob("figure_data_*.csv"):
    shutil.copy2(src, repo_fig / src.name)
for src in list(MANUSCRIPT_DIR.glob("*.md")) + [REPOSITORY_DIR / "README.md", REPOSITORY_DIR / "RUN_ORDER.md"]:
    if Path(src).exists():
        shutil.copy2(src, repo_docs / Path(src).name)
rows = []
for root in [Path("."), Path("/mnt/data")]:
    if root.exists():
        for p in root.glob("*.ipynb"):
            if p.is_file() and ".ipynb_checkpoints" not in str(p):
                dst = repo_nbs / p.name
                try:
                    shutil.copy2(p, dst)
                    status = "copied"
                except Exception as e:
                    status = f"copy_failed: {e}"
                rows.append({"notebook": str(p), "destination": str(dst), "status": status})
pd.DataFrame(rows).drop_duplicates().to_csv(AUDIT_DIR / "notebook_copy_audit.csv", index=False)
print("Repository content copied.")


Repository content copied.


In [10]:
# Final audit, manual tracker, manifest, and final zip
checklist = pd.DataFrame([
    {"category": "Manuscript", "item": "Replace author names/affiliations", "status": "pending"},
    {"category": "Repository", "item": "Add final GitHub/Zenodo URL", "status": "pending"},
    {"category": "Data", "item": "Add final data DOI", "status": "pending"},
    {"category": "Figures", "item": "Create final journal-quality Figure 1 and export figures", "status": "pending"},
    {"category": "Claim control", "item": "CDE evidence described as text-mined evidence only", "status": "complete"},
    {"category": "Claim control", "item": "C-strict described as post-DFT decision-support model", "status": "complete"},
])
checklist.to_csv(AUDIT_DIR / "next_sustainability_submission_checklist.csv", index=False)
tracker_text = """FINAL REPOSITORY UPLOAD CHANGE TRACKER
=====================================

A. MUST EDIT BEFORE PUBLIC UPLOAD
[ ] Replace placeholder author names.
[ ] Replace placeholder affiliations.
[ ] Replace corresponding author name and email.
[ ] Replace REPOSITORY_URL after GitHub/Zenodo upload.
[ ] Replace DATA_REPOSITORY_DOI after data deposit.
[ ] Confirm funding statement.
[ ] Confirm competing interests statement.
[ ] Confirm CRediT author contributions.
[ ] Confirm generative AI declaration.
[ ] Add LICENSE file.
[ ] Add final Figure 1 workflow diagram.
[ ] Convert HTML figures to journal-quality PNG/TIFF/PDF if required.
[ ] Check final abstract word count.

B. DO NOT UPLOAD
[ ] Do not upload Materials Project API key.
[ ] Do not upload .env files with secrets.
[ ] Do not upload old inflated outputs: mp_na_electrodes_master_v2_with_ml_predictions.csv, candidate_shortlist_for_manuscript.csv, family_level_summary.csv.
[ ] Do not upload __pycache__ or .ipynb_checkpoints folders.

C. SCIENTIFIC CLAIM CONTROL
Use: decision-support workflow; computed sodium-ion electrode records; text-mined literature evidence; screening-based shortlist; post-DFT decision-support model.
Avoid: experimentally validated candidates; new cathode discovered; CDE confirms performance; ML discovers materials; guaranteed sustainable.

D. REQUIRED INPUT BEFORE NOTEBOOK 05
[ ] outputs 4.zip from Notebook 04

E. FINAL PACKAGE LOCATION
sodium_cathode_submission_package_FINAL.zip
"""
tracker_path = AUDIT_DIR / "FINAL_REPOSITORY_UPLOAD_CHANGE_TRACKER.txt"
tracker_path.write_text(tracker_text, encoding="utf-8")
shutil.copy2(tracker_path, REPOSITORY_DIR / "FINAL_REPOSITORY_UPLOAD_CHANGE_TRACKER.txt")
manifest = []
for root, name in [(REPOSITORY_DIR, "repository_package"), (MANUSCRIPT_DIR, "manuscript_package"), (SUBMISSION_DIR, "submission_files"), (AUDIT_DIR, "audit")]:
    for p in Path(root).rglob("*"):
        if p.is_file():
            manifest.append({"package": name, "relative_path": str(p.relative_to(OUTPUT_DIR)), "size_kb": round(p.stat().st_size / 1024, 2)})
pd.DataFrame(manifest).to_csv(AUDIT_DIR / "final_package_manifest.csv", index=False)
FINAL_ZIP = Path("sodium_cathode_submission_package_FINAL.zip")
if FINAL_ZIP.exists():
    FINAL_ZIP.unlink()
with zipfile.ZipFile(FINAL_ZIP, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(OUTPUT_DIR.rglob("*")):
        if p.is_file():
            zf.write(p, p.relative_to(OUTPUT_DIR.parent))
required = [REPOSITORY_DIR / "README.md", REPOSITORY_DIR / "requirements.txt", REPOSITORY_DIR / "environment.yml", REPOSITORY_DIR / "RUN_ORDER.md", MANUSCRIPT_DIR / "manuscript_outline.md", MANUSCRIPT_DIR / "draft_abstract.md", MANUSCRIPT_DIR / "figure_captions.md", MANUSCRIPT_DIR / "table_captions.md", MANUSCRIPT_DIR / "claim_control_notes.md", SUBMISSION_DIR / "data_availability_statement.md", SUBMISSION_DIR / "code_availability_statement.md", SUBMISSION_DIR / "generative_ai_declaration.md", AUDIT_DIR / "next_sustainability_submission_checklist.csv", AUDIT_DIR / "final_package_manifest.csv", tracker_path, FINAL_ZIP]
audit = pd.DataFrame([{"file": str(p), "exists": Path(p).exists(), "size_kb": round(Path(p).stat().st_size / 1024, 2) if Path(p).exists() else 0} for p in required])
audit.to_csv(AUDIT_DIR / "notebook05_final_audit.csv", index=False)
display(audit)
missing = audit[audit["exists"] == False]
if len(missing):
    raise FileNotFoundError("Some final files are missing")
print("Notebook 05 completed successfully.")
print("Final package:", FINAL_ZIP.resolve())


,file,exists,size_kb
0,sodium_cathode_submission_package\outputs\repo...,True,1.86
1,sodium_cathode_submission_package\outputs\repo...,True,0.09
2,sodium_cathode_submission_package\outputs\repo...,True,0.27
3,sodium_cathode_submission_package\outputs\repo...,True,0.37
4,sodium_cathode_submission_package\outputs\manu...,True,0.68
5,sodium_cathode_submission_package\outputs\manu...,True,1.12
6,sodium_cathode_submission_package\outputs\manu...,True,0.50
7,sodium_cathode_submission_package\outputs\manu...,True,0.23
8,sodium_cathode_submission_package\outputs\manu...,True,0.41
9,sodium_cathode_submission_package\outputs\subm...,True,0.41


Notebook 05 completed successfully.
Final package: C:\Users\IICT-1\next sustainability\Final for github\sodium_cathode_submission_package_FINAL.zip


In [ ]:
from pathlib import Path
import zipfile
import pandas as pd

OUTPUT_DIR = Path("sodium_cathode_submission_package/outputs")
FINAL_ZIP = Path("sodium_cathode_submission_package_FINAL.zip")

required_final_files = [
    OUTPUT_DIR / "repository_package" / "README.md",
    OUTPUT_DIR / "repository_package" / "requirements.txt",
    OUTPUT_DIR / "repository_package" / "environment.yml",
    OUTPUT_DIR / "repository_package" / "RUN_ORDER.md",
    OUTPUT_DIR / "manuscript_package" / "manuscript_outline.md",
    OUTPUT_DIR / "manuscript_package" / "draft_abstract.md",
    OUTPUT_DIR / "manuscript_package" / "figure_captions.md",
    OUTPUT_DIR / "manuscript_package" / "table_captions.md",
    OUTPUT_DIR / "manuscript_package" / "title_page_draft.md",
    OUTPUT_DIR / "manuscript_package" / "claim_control_notes.md",
    OUTPUT_DIR / "submission_files" / "data_availability_statement.md",
    OUTPUT_DIR / "submission_files" / "code_availability_statement.md",
    OUTPUT_DIR / "submission_files" / "competing_interests_statement.md",
    OUTPUT_DIR / "submission_files" / "funding_statement.md",
    OUTPUT_DIR / "submission_files" / "generative_ai_declaration.md",
    OUTPUT_DIR / "audit" / "next_sustainability_submission_checklist.csv",
    OUTPUT_DIR / "audit" / "final_package_manifest.csv",
]

audit_rows = []

for p in required_final_files:
    audit_rows.append({
        "file": str(p),
        "exists": p.exists(),
        "size_kb": round(p.stat().st_size / 1024, 2) if p.exists() else 0,
    })

df_audit = pd.DataFrame(audit_rows)
audit_path = OUTPUT_DIR / "audit" / "notebook05_final_audit.csv"
df_audit.to_csv(audit_path, index=False)

if FINAL_ZIP.exists():
    FINAL_ZIP.unlink()

with zipfile.ZipFile(FINAL_ZIP, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(OUTPUT_DIR.rglob("*")):
        if p.is_file():
            zf.write(p, p.relative_to(OUTPUT_DIR.parent))

print("Rebuilt:", FINAL_ZIP)
print("Final audit saved:", audit_path)
display(df_audit)